In [1]:
# 1단계: 기존 구매 CSV 읽기 (상품 CSV 수정·재생성 불필요)
import pandas as pd

purchase_df = pd.read_csv('data/dummy_purchase_history.csv', encoding='utf-8-sig')

required_columns = {'member_id', 'category', 'quantity'}
missing = required_columns - set(purchase_df.columns)
if missing:
    raise ValueError(f'구매 CSV에 필요한 컬럼이 없습니다: {sorted(missing)}')

CATEGORIES = {
    'GPU', 'CASE', 'CPU', 'KEYBOARD', 'MOUSE', 'SSD',
    'SPEAKER', 'HDD', 'POWER', 'RAM', 'MONITOR', 'MAINBOARD'
}

purchase_df['category'] = purchase_df['category'].astype(str).str.strip().str.upper()
if purchase_df.empty:
    raise ValueError('구매 데이터가 비어 있습니다.')
if purchase_df['member_id'].isna().any():
    raise ValueError('member_id가 비어 있는 구매 기록이 있습니다.')
if not purchase_df['category'].isin(CATEGORIES).all():
    invalid = sorted(set(purchase_df['category']) - CATEGORIES)
    raise ValueError(f'알 수 없는 카테고리가 있습니다: {invalid}')

purchase_df['quantity'] = pd.to_numeric(purchase_df['quantity'], errors='raise')
if purchase_df['quantity'].isna().any() or (purchase_df['quantity'] <= 0).any():
    raise ValueError('quantity는 0보다 커야 합니다.')

purchase_df['purchased'] = 1
print('회원 수:', purchase_df['member_id'].nunique())
print('카테고리 수:', purchase_df['category'].nunique())
print('구매 기록 행 수:', len(purchase_df))
print(purchase_df[['member_id', 'category', 'quantity']].head())


회원 수: 1720
카테고리 수: 12
구매 기록 행 수: 13226
   member_id   category  quantity
0        611        RAM         1
1        611      POWER         1
2        611  MAINBOARD         1
3       1506        GPU         1
4       1506      POWER         1


In [2]:
# 2단계: 회원-카테고리 구매 여부 행렬 생성
# 같은 카테고리에서 여러 상품을 구매했어도 '해당 카테고리를 구매한 적 있음 = 1'로 처리합니다.
user_category_matrix = purchase_df.pivot_table(
    index='member_id',
    columns='category',
    values='purchased',
    aggfunc='max',
    fill_value=0
)

print('회원 x 카테고리 행렬:', user_category_matrix.shape)
print(user_category_matrix.iloc[:5, :])
assert 'product_id' not in user_category_matrix.columns, '상품 ID가 학습 특징으로 들어갔습니다.'


회원 x 카테고리 행렬: (1720, 12)
category   CASE  CPU  GPU  HDD  KEYBOARD  MAINBOARD  MONITOR  MOUSE  POWER  \
member_id                                                                    
1             0    1    1    0         1          1        1      1      0   
3             0    1    1    0         0          1        0      0      0   
4             0    0    0    0         0          0        0      0      0   
6             0    0    0    0         0          0        1      0      0   
7             1    0    0    1         0          0        1      0      0   

category   RAM  SPEAKER  SSD  
member_id                     
1            0        0    1  
3            1        0    1  
4            0        0    1  
6            0        1    1  
7            0        0    1  


In [3]:
# 3단계: 카테고리 간 구매 패턴 유사도 학습
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

category_user_matrix = user_category_matrix.T
similarity = cosine_similarity(category_user_matrix)
# 같은 카테고리 자기 자신과의 유사도(1.0)는 추천 점수에 반영하지 않습니다.
np.fill_diagonal(similarity, 0.0)

category_similarity_df = pd.DataFrame(
    similarity,
    index=category_user_matrix.index,
    columns=category_user_matrix.index
)

# 신규 회원의 인기 카테고리 추천용 구매 수량 총합
popular_categories = (
    purchase_df.groupby('category')['quantity']
    .sum()
    .sort_values(ascending=False)
)

print('카테고리 유사도 행렬:', category_similarity_df.shape)
print(category_similarity_df.iloc[:5, :5].round(3))
print('인기 카테고리:', popular_categories.head(5).to_dict())


카테고리 유사도 행렬: (12, 12)
category   CASE    CPU    GPU    HDD  KEYBOARD
category                                      
CASE      0.000  0.518  0.578  0.442     0.543
CPU       0.518  0.000  0.623  0.482     0.546
GPU       0.578  0.623  0.000  0.435     0.544
HDD       0.442  0.482  0.435  0.000     0.477
KEYBOARD  0.543  0.546  0.544  0.477     0.000
인기 카테고리: {'MONITOR': 1779, 'CPU': 1570, 'KEYBOARD': 1546, 'GPU': 1378, 'SSD': 1335}


In [4]:
# 4단계: 카테고리 추천 함수 (상품 ID는 사용하지 않음)
def recommend_categories(purchased_categories, top_n=5):
    # purchased_categories는 실제 회원의 구매 이력에서 얻은 카테고리 목록입니다.
    # 예: ['CPU', 'RAM'] / 신규 회원: []
    categories = [str(x).strip().upper() for x in purchased_categories]
    unknown = sorted(set(categories) - set(category_similarity_df.index))
    if unknown:
        raise ValueError(f'모델에 없는 카테고리입니다: {unknown}')

    if not categories:
        # 신규 회원: 가장 많이 팔린 카테고리 추천
        scores = popular_categories.astype(float)
        scores = scores / scores.max()
        reason = 'POPULAR'
    else:
        # 기존 회원: 구매 카테고리와 함께 구매되는 다른 카테고리들의 유사도 합산
        scores = pd.Series(0.0, index=category_similarity_df.index)
        for category in set(categories):
            scores += category_similarity_df[category]
        scores = scores[scores > 0].sort_values(ascending=False)
        if scores.empty:
            # 카테고리 공동 구매 기록이 없는 예외 상황
            scores = popular_categories.astype(float)
            scores = scores / scores.max()
            reason = 'POPULAR'
        else:
            scores = scores / scores.max()
            reason = 'PERSONALIZED'

    best = scores.sort_values(ascending=False).head(top_n)
    return pd.DataFrame({
        'category': best.index.to_list(),
        'recommend_score': best.values,
        'recommend_type': reason
    })

# 학습용 회원 1명의 카테고리를 가져와 시험합니다. 회원 ID 자체는 모델에 저장하지 않습니다.
test_member_id = user_category_matrix.index[0]
purchased_categories = user_category_matrix.columns[
    user_category_matrix.loc[test_member_id] > 0
].tolist()
print('테스트 회원:', test_member_id)
print('구매 카테고리:', purchased_categories)
print('맞춤 추천:')
print(recommend_categories(purchased_categories, top_n=5).to_string(index=False))
print('신규 회원 인기 추천:')
print(recommend_categories([], top_n=5).to_string(index=False))


테스트 회원: 1
구매 카테고리: ['CPU', 'GPU', 'KEYBOARD', 'MAINBOARD', 'MONITOR', 'MOUSE', 'SSD']
맞춤 추천:
category  recommend_score recommend_type
   POWER         1.000000   PERSONALIZED
     RAM         0.995943   PERSONALIZED
    CASE         0.977455   PERSONALIZED
KEYBOARD         0.933945   PERSONALIZED
 MONITOR         0.919059   PERSONALIZED
신규 회원 인기 추천:
category  recommend_score recommend_type
 MONITOR         1.000000        POPULAR
     CPU         0.882518        POPULAR
KEYBOARD         0.869028        POPULAR
     GPU         0.774592        POPULAR
     SSD         0.750422        POPULAR


In [5]:
# 5단계: DB 상품 ID/가상 회원 ID가 저장되지 않는 모델 저장
from pathlib import Path
import joblib

Path('models').mkdir(exist_ok=True)
model = {
    'model_type': 'category_cf_v1',
    'category_similarity_df': category_similarity_df,
    'popular_categories': popular_categories,
}

# 학습 당시의 가상 회원·상품 ID가 저장되지 않았는지 확인
assert set(model.keys()) == {'model_type', 'category_similarity_df', 'popular_categories'}
assert set(category_similarity_df.index) == set(category_similarity_df.columns)
assert set(category_similarity_df.index) == set(popular_categories.index)

model_path = 'models/category_cf_model_pc12.pkl'
joblib.dump(model, model_path)
print('저장 완료:', model_path)
print('저장된 모델 구성:', list(model.keys()))


저장 완료: models/category_cf_model_pc12.pkl
저장된 모델 구성: ['model_type', 'category_similarity_df', 'popular_categories']


In [6]:
# 6단계: 저장된 모델 다시 읽어서 기본 검증
loaded = joblib.load('models/category_cf_model_pc12.pkl')
assert loaded['model_type'] == 'category_cf_v1'
assert loaded['category_similarity_df'].shape == (12, 12)
assert all(isinstance(cat, str) for cat in loaded['category_similarity_df'].index)
assert 'user_item_matrix' not in loaded
assert 'product_df' not in loaded
print('검증 통과: 12개 카테고리 기준, 상품/회원 ID 없는 모델')


검증 통과: 12개 카테고리 기준, 상품/회원 ID 없는 모델
